In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
PATH = "/kaggle/input/competitions/ieee-fraud-detection"
id= pd.read_csv(PATH+"/train_identity.csv")

In [ ]:
id


In [ ]:
tr = pd.read_csv(PATH+"/train_transaction.csv")

In [ ]:
tr


In [ ]:
tr.head()


In [ ]:
  !pip install mlflow dagshub


In [ ]:
# import dagshub
# dagshub.init(repo_owner='lukaLomadze', repo_name='ML_Assignment_2', mlflow=True)

# import mlflow
# with mlflow.start_run():
#   mlflow.log_param('parameter name', 'value')
#   mlflow.log_metric('metric name', 1)

In [ ]:
# t = tr.merge(id, on="TransactionID", how="left")


In [ ]:
# t

In [ ]:
# tt = tr.merge(id, on="TransactionID")

In [ ]:
# tt

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
import mlflow.sklearn

In [ ]:
TARGET= "isFraud"
TR_MISS_TH= 0.8
ID_MISS_TH = 0.7
CARDINALITY_TH=6
CORRELATION_TH=0.8


In [ ]:
import mlflow
import dagshub
dagshub.init(repo_owner='lukaLomadze', repo_name='ML_Assignment_2', mlflow=True)
mlflow.set_tracking_uri("https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow")
mlflow.set_experiment('Logistic_regression')

# **Cleaning**


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
class DropHighMissing(BaseEstimator, TransformerMixin):
    def __init__(self, tr_th=TR_MISS_TH, id_th=ID_MISS_TH, identity_cols=None):
        self.tr_th = tr_th
        self.id_th = id_th
        self.identity_cols = identity_cols

    def fit(self, X, y=None):
        id_set = set(self.identity_cols) if self.identity_cols else set()
        keep = []
        for col in X.columns:
            miss_rate = X[col].isnull().mean()
            threshold = self.id_th if col in id_set else self.tr_th
            if miss_rate <= threshold:
                keep.append(col)
        self.cols_ = keep
        return self

    def transform(self, X):
        return X[self.cols_]

In [ ]:
tr.shape


In [ ]:
id.shape

In [ ]:
train = tr.merge(id, on="TransactionID", how="left")
print(train.head())

In [ ]:
y = train[TARGET].copy()


In [ ]:
print("train", train.shape)

In [ ]:
cat_cols=train.select_dtypes(include='object').columns.tolist()
num_cols= train.select_dtypes(exclude='object').columns.tolist()

In [ ]:
with mlflow.start_run(run_name="lr_cleaning"):
    mlflow.log_param("TR_MISS_TH", TR_MISS_TH)
    mlflow.log_param("ID_MISS_TH", ID_MISS_TH)
    mlflow.log_param("Final shape", f"{train.shape[0]} X {train.shape[1]}")
    
    mlflow.log_param("Num cols", len(num_cols))
    mlflow.log_param("Catcols", len(cat_cols))
    mlflow.log_param("Fraud rate", float(y.mean()))
    

# Feature engineering

In [ ]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, identity_cols=None, identity_ids=None, target=None):
        self.identity_cols = identity_cols
        self.identity_ids  = identity_ids   
        self.target        = target

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if "TransactionAmt" in X:
            X["TransactionAmt_log"] = np.log1p(X["TransactionAmt"])
        if "TransactionDT" in X:
            X["TransactionHour"] = (X["TransactionDT"]// 3600) % 24
            X["TransactionDay"]  = (X["TransactionDT"] //86400) % 7

        if self.identity_ids is not None and "TransactionID" in X.columns:
            X["has_identity"] = X["TransactionID"].isin(self.identity_ids).astype(int)

        drop_cols = []
        if "TransactionID" in X.columns:
            drop_cols.append("TransactionID")
        if self.target and self.target in X.columns:
            drop_cols.append(self.target)
        if drop_cols:
            X = X.drop(columns=drop_cols)

        return X

In [ ]:
low_card_cols  = [c for c in cat_cols if train[c].nunique() <= CARDINALITY_TH]
high_card_cols = [c for c in cat_cols if train[c].nunique() >  CARDINALITY_TH]

In [ ]:
low_card_cols

In [ ]:
high_card_cols

In [ ]:
train.shape

In [ ]:
with mlflow.start_run(run_name="lr_feature_engineering"):
    mlflow.log_param("New features", "TransactionDay, TransactionHour ,TransactionAmt, Has_identity")
    mlflow.log_param("CARDINALITY_TH", CARDINALITY_TH) 
    mlflow.log_param("Low cardinality cols", len(low_card_cols))
    mlflow.log_param("High cardinality cols", len(high_card_cols))
   

# Feature selection

In [ ]:
class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, th=CORRELATION_TH):
        self.th = th

    def fit(self, X, y=None):
        num = X.select_dtypes(exclude="object")
        corr = num.fillna(num.median()).corr().abs() 
        # es zeda ori xazi yoveli shemxvevistvis wesit arasdors iqneba sachiro
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        self.drop_cols_ = [c for c in upper.columns if any(upper[c] > self.th)]
        return self

    def transform(self, X):
        return X.drop(columns=self.drop_cols_, errors="ignore")

In [ ]:
with mlflow.start_run(run_name="lr_feature_selection"):
    mlflow.log_param("Method", "corrrelatin_filter")
    mlflow.log_param("CORRELATION_TH", CORRELATION_TH) 
   

Preprocessor


In [ ]:
class Preprocessor(BaseEstimator, TransformerMixin):
   

    def __init__(self, cardinality_th=10):
        self.cardinality_th = cardinality_th

    def _build(self, X):
        num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
        cat_cols = X.select_dtypes(include="object").columns.tolist()

        low_card  = [c for c in cat_cols if X[c].nunique() <= self.cardinality_th]
        high_card = [c for c in cat_cols if X[c].nunique() >  self.cardinality_th]

        num_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler",  StandardScaler())
        ])
        ohe_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ])
        ord_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ])

        transformers = [("num", num_pipe, num_cols)]
        if low_card:
            transformers.append(("ohe", ohe_pipe, low_card))
        if high_card:
            transformers.append(("ord", ord_pipe, high_card))

        return ColumnTransformer(transformers, remainder="drop")

    def fit(self, X, y=None):
        self.ct_ = self._build(X)
        self.ct_.fit(X, y)
        return self

    def transform(self, X):
        return self.ct_.transform(X)

In [ ]:
def build_pipeline(C, penalty, solver, sampling, max_iter=500):

    sampler = SMOTE(random_state=42) if sampling == "smote" else RandomUnderSampler(random_state=42)

    identity_cols = [c for c in id.columns if c != "TransactionID"]
    identity_ids  = set(id["TransactionID"].unique())

    return ImbPipeline([
        ("drop",   DropHighMissing(tr_th=TR_MISS_TH, id_th=ID_MISS_TH, identity_cols=identity_cols)),
        ("feat",   FeatureEngineer(identity_cols=identity_cols, identity_ids=identity_ids, target=TARGET)),
        ("select", CorrelationFilter(CORRELATION_TH)),
        ("prep",   Preprocessor()),
        ("sample", sampler),
        ("model",  LogisticRegression(C=C, penalty=penalty, solver=solver, max_iter=max_iter, n_jobs=-1))
    ])

Training

In [ ]:
PARAM_GRID = [
    dict(C=0.001, penalty="l2", solver="lbfgs", sampling="smote"),
    dict(C=0.01,  penalty="l2", solver="lbfgs", sampling="smote"),
    dict(C=0.1,   penalty="l2", solver="lbfgs", sampling="smote"),
    dict(C=1.0,   penalty="l2", solver="lbfgs", sampling="smote"),
    dict(C=0.1,   penalty="l1", solver="liblinear", sampling="smote"),
    dict(C=0.1,   penalty="l2", solver="lbfgs", sampling="undersample"),
]

CV = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

best_auc = -1
best_params = None
results = []

In [ ]:
X=train

In [ ]:
for p in PARAM_GRID:
    with mlflow.start_run(run_name=f"LR_c={p["C"]} , pen={p["penalty"]} solver ={p["solver"]}, sapling={p["sampling"]}"):

        pipe = build_pipeline(p["C"], p["penalty"], p["solver"], p["sampling"])

        scores = cross_val_score(pipe, X, y, cv=CV, scoring="roc_auc", n_jobs=-1)
        mean_auc = scores.mean()
        std_auc  = scores.std()

        mlflow.log_params(p)
        mlflow.log_metric("cv_auc_mean", mean_auc)
        mlflow.log_metric("cv_auc_std",  std_auc)

        results.append((p, mean_auc))

        if mean_auc > best_auc:
            best_auc    = mean_auc
            best_params = p

In [ ]:

print(f"Best params: {best_params}  |  CV AUC: {best_auc}")

best_pipe = build_pipeline(
    C        = best_params["C"],
    penalty  = best_params["penalty"],
    solver   = best_params["solver"],
    sampling = best_params["sampling"],
)

with mlflow.start_run(run_name="LR_best_registered") as run:
    mlflow.log_params(best_params)
    mlflow.log_metric("cv_auc_mean", best_auc)

   
    best_pipe.fit(X, y)

    
    model_info = mlflow.sklearn.log_model(
        sk_model        = best_pipe,
        artifact_path   = "model",
        registered_model_name = "LR_Best",
    )

    print(f"Model registered: {model_info.model_uri}")
    print(f"Run ID: {run.info.run_id}")